# 10.5 怎麼學到可用的視覺特徵？

# 第 10 章：圖片如何進入文字模型

前置：embedding、序列與Linear。圖片入口先用16×16合成RGB，不需外部權重；圖片是否被理解必須等語言／視覺基礎能力訓練後測試。程式在tiny_perceptron/multimodal.py。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：圖片切小方塊：先檢查能不能拼回來**

16×16 RGB，patch邊長4，得到16個patch。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/patchify.svg")))

**先想一想：**隨機encoder的shape正確，就代表知道圓形嗎？

先教encoder辨別生成器已知的形狀／顏色，才能談投影到語言模型。小分類head提供可核對label，之後丟掉head保留features。

**把直覺寫成數學：**分類CE的梯度回到encoder；train/held-out按屬性組合或生成seed分開。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.multimodal import VisionEncoder, scene

encoder = VisionEncoder(width=8)
classifier = nn.Linear(8, 2)
images = torch.stack([scene("red", "square"), scene("blue", "circle")])
features = encoder(images)
logits = classifier(features.mean(1))
loss = F.cross_entropy(logits, torch.tensor([0, 1]))
loss.backward()
print(features.shape, loss.item(), encoder.projection.weight.grad.norm().item())

**如何讀結果：**這裏只核對監督與梯度。GPU訓練入口是scripts/pretrain_encoders.py --modality vision --train。

**只改一件事：**只打亂shape labels，說明這將教錯什麼。
